# STARRT: fifth GPU round (hyper-parameter tuning)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. *Add Input* → the `starrt-gpu5` dataset.

Run cell 1 and the smoke test first. Then **Save Version → Save & Run All (Commit)**. Total about 4-5 hours. Results → `starrt_gpu5_results.zip`.

In [ ]:
import glob, os, subprocess
hits = glob.glob('/kaggle/input/**/tune_models.py', recursive=True)
assert hits, 'Dataset not attached: Add Input -> starrt-gpu5'
CODE = os.path.dirname(hits[0]); print('code folder:', CODE)
import torch; print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run(script, args=''):
    cmd = f'cd /kaggle/working && python -u {CODE}/{script} {args}'
    print('>>', cmd, flush=True); r = subprocess.run(cmd, shell=True)
    if r.returncode != 0: print('!! FAILED:', script, args, '(continuing with the next step)')

## 0. Smoke test (5 min)

In [ ]:
run('tune_models.py', '--family distilbert --datasets faruque --limit 40 --model distilbert-base-uncased')
print('smoke test done: check above that it printed [distilbert_tuned | faruque] lines')
import shutil; shutil.rmtree('/kaggle/working/results', ignore_errors=True)

## A. Sentence-BERT + LR: tune C (nested, ~25 min)

In [ ]:
for m in ['sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-mpnet-base-v2']:
    run('tune_models.py', f'--family sbert --model {m} --datasets djinni,faruque,stackoverflow')

## B. DistilBERT: tune learning rate x epochs (nested), Faruque then Djinni (~4 h)

In [ ]:
run('tune_models.py', '--family distilbert --datasets faruque,djinni')

## Zip results
Download `starrt_gpu5_results.zip` from the **Output** tab and send it in the chat.

In [ ]:
import shutil
for f in glob.glob('/kaggle/working/results/emb_*.npy'): os.remove(f)
shutil.make_archive('/kaggle/working/starrt_gpu5_results', 'zip', '/kaggle/working/results')
print(sorted(os.listdir('/kaggle/working/results')))